In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import numpy as np
PROJECT_ROOT = os.path.abspath("..") 
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_final import TraceOttoDataset
from model.trace import TRACE
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
from utils.training_utils import update_binary_metrics
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
trace_model = TRACE(
    num_embeddings_aid=1855603,
    num_embeddings_event_type=4,
    embedding_dim=32,
    num_classes=2
).to(device)

checkpoint = torch.load("Model_TRACE_ATC_FinalVersion_SingleTask.pt", weights_only=False, map_location=device)

trace_model.load_state_dict(checkpoint["model_state_dict"])
best_thr_ATC = checkpoint["best_global_threshold"]["ATC"]
best_thr_SAT = checkpoint["best_global_threshold"]["SAT"]

trace_model.eval()
print(f"Loaded model Optimum threshold ATC: {best_thr_ATC:.4f} SAT: {best_thr_SAT:.4f}")

In [ ]:
dataset_processed  = TraceOttoDataset(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=16,
)

In [ ]:
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=32)


print(len(train_loader.dataset))
print(len(validation_loader.dataset))
print(len(test_loader.dataset))


In [ ]:
trace_model.eval()
criterion = torch.nn.BCEWithLogitsLoss()
test_loss = 0.0
test_correct_ATC = 0
test_total_ATC = 0
test_correct_SAT = 0
test_total_SAT = 0

y_true_ATC = []
y_pred_ATC = []
y_true_SAT = []
y_pred_SAT = []
with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        label_test_task_ATC = targets_test["ATC"].unsqueeze(1).to(device).float()
        label_test_task_SAT = targets_test["SAT"].unsqueeze(1).to(device).float()
        inputs_test = {k: v.to(device) for k, v in inputs_test.items()}

        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device).float()
        delta_between = get_between_features(inputs_test["timestamps"]).to(device).float()

        logits = trace_model(
            inputs_test["aid"],
            inputs_test["type"],
            delta_elapsed,
            delta_between
        )

        loss_ATC = criterion(logits[:, 0:1], label_test_task_ATC)
        loss_SAT = criterion(logits[:, 1:2], label_test_task_SAT)
        
        loss_test = loss_ATC + loss_SAT
        test_loss += loss_test.item()
        
        test_correct_ATC, test_total_ATC = update_binary_metrics(
            logits[:, 0:1],
            label_test_task_ATC,
            test_correct_ATC,
            test_total_ATC,
            y_true_ATC,
            y_pred_ATC,
            threshold=best_thr_ATC
        )

        test_correct_SAT, test_total_SAT = update_binary_metrics(
            logits[:, 1:2],
            label_test_task_SAT,
            test_correct_SAT,
            test_total_SAT,
            y_true_SAT,
            y_pred_SAT,
            threshold=best_thr_SAT
        )
        

test_loss /= len(test_loader)
test_accuracy_ATC = test_correct_ATC / max(test_total_ATC, 1)
test_accuracy_SAT = test_correct_SAT / max(test_total_SAT, 1)


y_true_ATC = torch.cat(y_true_ATC).numpy().astype(int).ravel()
y_pred_ATC = torch.cat(y_pred_ATC).numpy().astype(int).ravel()

y_true_SAT = torch.cat(y_true_SAT).numpy().astype(int).ravel()
y_pred_SAT = torch.cat(y_pred_SAT).numpy().astype(int).ravel()


f1_ATC = f1_score(y_true_ATC, y_pred_ATC)
f1_SAT = f1_score(y_true_SAT, y_pred_SAT)


f1_mean = (f1_ATC + f1_SAT) / 2
cm_ATC = confusion_matrix(y_true_ATC, y_pred_ATC)
cm_SAT = confusion_matrix(y_true_SAT, y_pred_SAT)
test_accuracy_global = (test_correct_ATC + test_correct_SAT) / max(test_total_ATC + test_total_SAT, 1)

print(f"Test accuracy global: {test_accuracy_global:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {np.mean([test_accuracy_ATC, test_accuracy_SAT]):.4f}")
print(f"F1 Mean Score: {f1_mean}")
print(f"F1 ATC Score: {f1_ATC}")
print(f"F1 SAT Score: {f1_SAT}")
print(f"Optimum Threshold {best_thr_ATC:.4f}")
print(f"Optimum Threshold {best_thr_SAT:.4f}")
print("Confusion Matrix: ATC\n", cm_ATC)
print("Confusion Matrix: SAT\n", cm_SAT)

print("\nClassification Report ATC:")
print(classification_report(y_true_ATC, y_pred_ATC, target_names=["ATC 0", "ATC 1"]))

print("\nClassification Report SAT:")
print(classification_report(y_true_SAT, y_pred_SAT, target_names=["SAT 0", "SAT 1"]))


fig1 = plot_confusion_matrix(
    cm_ATC,
    name_task="ATC",
    name_classes=["ATC 0", "ATC 1"]
)
fig2 = plot_confusion_matrix(
    cm_SAT,
    name_task="SAT",
    name_classes=["SAT 0", "SAT 1"]
)
plt.show()
plt.close(fig1)
plt.close(fig2)
